In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from sklearn.metrics import confusion_matrix
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from Model import ASLDataset, CNN


dataset_classes = sorted(['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 
			'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 
			'del', 'nothing', 'space'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_df_path = "../Data/train_path_dataframe.csv"
best_model_path = "../models/best.pt"
last_model_path = "../models/last.pt"

IMAGE_SIZE = 64
BATCH_SIZE = 64
device

In [ ]:
train_path_df = pd.read_csv(train_df_path)

In [ ]:
# Training transforms with Augmentation (Bonus)
train_transforms = transforms.Compose([
	transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
	transforms.RandomRotation(10),           # Randomly rotate the image by up to 15 degrees
	transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue = 0.2),  # Randomly change brightness
	transforms.RandomGrayscale(p=0.2), # Randomly transform to black and white
	transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)), # Randomly shift the image
	transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)), # random blur
	transforms.ToTensor()
 	# transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # برای درگیر میزان نور نشدن 
])

# train_transforms = transforms.Compose([
#     # ۱. اندازه دست مهم نیست (تغییر مقیاس تصادفی برای بی‌تفاوت کردن مدل به کوچک/بزرگ بودن دست)
#     transforms.RandomResizedCrop(
#         IMAGE_SIZE, 
#         scale=(0.7, 1.0),      # دست می‌تواند ۷۰٪ تا ۱۰۰٪ کادر را پر کند
#         ratio=(0.9, 1.1)
#     ),
    
#     # ۲. جایگاه دست مهم نیست (انتقال، چرخش و زاویه تصادفی)
#     transforms.RandomRotation(15),
#     transforms.RandomAffine(degrees=0, translate=(0.15, 0.15)), # جابجایی دست در کادر
    
#     # ۳. رنگ مهم نیست (تبدیل مستقیم به Grayscale با ۳ کانال جهت سازگاری با CNN)
#     transforms.Grayscale(num_output_channels=3),
    
#     # ۴. میزان نور مهم نیست (تغییرات شدید کنتراست و روشنایی روی تصویر سیاه و سفید)
#     transforms.ColorJitter(brightness=0.5, contrast=0.5),
    
#     transforms.ToTensor(),
    
#     # ۵. استخراج بهتر لبه‌ها و خنثی‌سازی نوسانات کلی نور
#     transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
# ])

# Validation/Test transforms (Clean data)
test_transforms = transforms.Compose([
	transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    	transforms.Grayscale(num_output_channels=3),
	transforms.ToTensor()
 	# transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # برای درگیر میزان نور نشدن 
])

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(train_path_df['image_path'], train_path_df['label'], test_size = 0.2, random_state=54)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_train,  Y_train, test_size = 0.25, random_state=54)


In [ ]:
# 3. Instantiate the Train, Validation, and Test Datasets
train_dataset = ASLDataset(X_train, Y_train, transform=train_transforms)
valid_dataset = ASLDataset(X_valid, Y_valid, transform=test_transforms)
test_dataset = ASLDataset(X_test, Y_test, transform=test_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 9)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)

In [ ]:
def learning_process(
        model: CNN,
        train_loader: DataLoader = train_loader,
        valid_loader: DataLoader = valid_loader,
        criterion: nn.CrossEntropyLoss = nn.CrossEntropyLoss(),
        learning_rate = 0.0001,
        epochs = 13
        ):
        # optimizer = optim.Adam(model.parameters(), lr=learning_rate)
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
        
        train_losses, valid_losses = [], []
        train_accs, valid_accs = [], []
        
        best_val_loss = float('inf') # متغیری برای ذخیره بهترین Loss
        
        # Training
        for epoch in range(epochs):
                print(f"------------ {epoch = } Started")
                # Set model to training mode (enables Dropout and BatchNorm updates)
                model.train()
                
                running_loss, correct, total = 0.0, 0, 0
                for i, (images, labels) in enumerate(train_loader):
                        images, labels = images.to(device), labels.to(device)
                        # Clear old gradients
                        optimizer.zero_grad()
                        # Forward pass: predict labels
                        outputs = model(images)
                        # Calculate error
                        loss = criterion(outputs, labels)
                        # Backward pass: compute gradients
                        loss.backward()
                        # Update weights
                        optimizer.step()
                        
                        running_loss += loss.item()
                        _, predicted = torch.max(outputs.data, 1)
                        total += labels.size(0)
                        correct += (predicted == labels).sum().item()
                        
                train_losses.append(running_loss / len(train_loader))
                train_accs.append(100 * correct / total)
                
                model.eval()
                valid_loss, correct, total = 0.0, 0, 0
                
                with torch.no_grad():
                        for images, labels in valid_loader:
                                images, labels = images.to(device), labels.to(device)
				# Forward pass: predict labels
                                outputs = model(images)
				# Calculate error
                                loss = criterion(outputs, labels)
				
                                valid_loss += loss.item()
                                _, predicted = torch.max(outputs, 1)
                                total += labels.size(0)
                                correct += (predicted == labels).sum().item()
                valid_losses.append(valid_loss / len(valid_loader))
                valid_accs.append(100 * correct / total)
                
                print(f"Epoch {epoch+1}/{epochs} | "
                      f"Train Loss: {train_losses[-1]:.4f}, Acc: {train_accs[-1]:.2f}% | "
                      f"Val Loss: {valid_losses[-1]:.4f}, Acc: {valid_accs[-1]:.2f}%")
                
                if valid_losses[-1] < best_val_loss:
                        best_val_loss = valid_losses[-1]
                        torch.save(model.state_dict(), best_model_path)
                        print(f"Best model saved with Val Loss: {best_val_loss:.4f}")
        return train_losses, train_accs, valid_losses, valid_accs

In [ ]:
model = CNN(num_classes=29).to(device)

train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, epochs=10)

In [ ]:
torch.save(model.state_dict(), last_model_path)
print("Model saved successfully!")

In [ ]:
# How to load:
model = CNN(num_classes=29).to(device)
model.load_state_dict(torch.load(best_model_path))
model.eval()

In [ ]:
def test_model(model, test_loader = test_loader):
	eval_model = model # or loaded_model

	# 2. Set the model to evaluation mode (Crucial: turns off Dropout and BatchNorm updates)
	eval_model.eval()

	# 3. Define the loss function (since it was previously trapped inside your learning_process function)
	criterion = nn.CrossEntropyLoss()

	test_loss = 0.0
	correct = 0
	total = 0

	# Lists to store all predictions and true labels for the Confusion Matrix
	all_preds = []
	all_true = []

	print("Starting evaluation on the Test Dataset...")

	# Turn off gradient calculation to save memory and speed up inference
	with torch.no_grad():
		for images, labels in test_loader:
			# Move data to the same device as the model (cuda or cpu)
			images, labels = images.to(device), labels.to(device)
			
			# Forward pass
			outputs = eval_model(images)
			loss = criterion(outputs, labels)
			
			# Track Loss and Accuracy
			test_loss += loss.item()
			_, predicted = torch.max(outputs.data, 1)
			
			total += labels.size(0)
			correct += (predicted == labels).sum().item()
			
			# Store predictions and true labels for the matrix
			all_preds.extend(predicted.cpu().numpy())
			all_true.extend(labels.cpu().numpy())

	# Calculate final averages
	avg_test_loss = test_loss / len(test_loader)
	test_accuracy = 100 * correct / total

	print("-" * 30)
	print(f"Final Test Loss: {avg_test_loss:.4f}")
	print(f"Final Test Accuracy: {test_accuracy:.2f}%")
	print("-" * 30)

	# 4. Generate and Plot the Confusion Matrix
	cm = confusion_matrix(all_true, all_preds)

	# ۱. محاسبه درصد هر خانه نسبت به جمع سطر (true label)
	row_sums = cm.sum(axis=1, keepdims=True)
	# جلوگیری از تقسیم بر صفر (اگر سطری هیچ نمونه‌ای نداشت)
	row_sums[row_sums == 0] = 1
	percent = (cm / row_sums) * 100

	# ۲. ساخت ماتریس کدهای عددی (برای اعمال رنگ‌بندی)
	color_indices = np.zeros_like(cm, dtype=int)

	# ۳. ساخت ماتریس متن برای نمایش داخل خانه‌ها
	text_matrix = np.empty_like(cm, dtype='<U20')
	for i in range(cm.shape[0]):
		for j in range(cm.shape[1]):
			p = percent[i, j]
			text_matrix[i, j] = f"{cm[i, j]}"
		
			# --- اعمال قوانین رنگ‌بندی ---
			if i == j:  # خانه روی قطر اصلی
				if p > 75:
					color_indices[i, j] = 1   # سبز پررنگ
				elif p > 50:
					color_indices[i, j] = 2   # سبز کمرنگ
				elif p > 25:
					color_indices[i, j] = 3   # قرمز کمرنگ
				else:
					color_indices[i, j] = 4   # قرمز پررنگ
			else:  # خانه خارج از قطر اصلی
				if p > 20:
					color_indices[i, j] = 5   # قرمز پررنگ
				elif p > 5:
					color_indices[i, j] = 6   # قرمز کمرنگ
				else:
					color_indices[i, j] = 7   # سفید (بدون رنگ)

	# ۴. تعریف نگاشت کدهای عددی به رنگ‌های هگز
	colors_map = {
		1: '#2ecc71',   # سبز پررنگ (قطر > ۷۵%)
		2: '#a8e6cf',   # سبز کمرنگ (قطر ۵۰% تا ۷۵%)
		3: '#ffb3b3',   # قرمز کمرنگ (قطر ۲۵% تا ۵۰%)
		4: '#e74c3c',   # قرمز پررنگ (قطر < ۲۵%)
		5: '#e74c3c',   # قرمز پررنگ (غیرقطر > ۲۰%)
		6: '#ffb3b3',   # قرمز کمرنگ (غیرقطر ۵% تا ۲۰%)
		7: '#ffffff'    # سفید (غیرقطر < ۵%)
	}

	# ۵. ساخت colorscale گسسته برای Plotly
	# (مقادیر z بین ۰.۵ تا ۷.۵ قرار می‌گیرند تا هر عدد صحیح یک بازهٔ اختصاصی داشته باشد)
	zmin, zmax = 0.5, 7.5
	colorscale = []
	for val in sorted(colors_map.keys()):
		low = (val - 0.5 - zmin) / (zmax - zmin)
		high = (val + 0.5 - zmin) / (zmax - zmin)
		colorscale.append([low, colors_map[val]])
		colorscale.append([high, colors_map[val]])

	# ۶. رسم هیت‌مپ با استفاده از color_indices به عنوان z
	fig = go.Figure(data=go.Heatmap(
		z=color_indices,                    # ماتریس کدهای عددی (۱ تا ۷)
		x=dataset_classes,                  # برچسب ستون‌ها (پیش‌بینی)
		y=dataset_classes,                  # برچسب سطرها (واقعیت)
		colorscale=colorscale,              # رنگ‌بندی گسستهٔ سفارشی
		zmin=zmin,
		zmax=zmax,
		text=text_matrix,                   # متن نمایشی داخل خانه‌ها
		texttemplate='%{text}',
		textfont={"size": 10, "color": "black"},
		hovertemplate='True: %{y}<br>Predicted: %{x}<br>%{text}<extra></extra>'
	))

	# ۷. تنظیمات ظاهری
	fig.update_layout(
		title='Confusion Matrix: Test Dataset',
		xaxis_title='Predicted Label',
		yaxis_title='True Label',
		width=900,
		height=700,
		xaxis={'tickangle': 45}
	)

	fig.show()

In [ ]:
test_model(model)

## Debug

In [ ]:
my_data_df = pd.read_csv("../Data2/my_data_path_dataframe.csv")

X_test2, Y_test2 = my_data_df['image_path'], my_data_df['label']
train2_dataset = ASLDataset(X_test2, Y_test2, transform = train_transforms)
test2_dataset = ASLDataset(X_test2, Y_test2, transform = test_transforms)
train2_loader = DataLoader(train2_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)
test2_loader = DataLoader(test2_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers = 3)

In [ ]:
model = CNN(num_classes=29).to(device)

# OP 0 (Normal)
# train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, epochs=10)

# OP 1
train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, epochs=10)
train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, train_loader = train2_loader, epochs=2)

# OP 2
# Same result as normal because training_loader is the same
# train_losses, train_accs, valid_losses, valid_accs = learning_process(model = model, valid_loader=test2_loader, epochs=10)

In [ ]:
test_model(model)

In [ ]:
test_model(model, test_loader = test2_loader)